In [1]:

import os
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"
os.environ["_JAVA_OPTIONS"] = "--add-opens=java.base/javax.security.auth=ALL-UNNAMED"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

# Verify Java is found
import subprocess
result = subprocess.run(["java", "-version"], capture_output=True, text=True)
print(result.stderr)  # java -version prints to stderr, this is normal

# Verify both files exist
import os.path
print("winutils.exe found:", os.path.isfile(r"C:\hadoop\bin\winutils.exe"))
print("hadoop.dll found:  ", os.path.isfile(r"C:\hadoop\bin\hadoop.dll"))

Picked up _JAVA_OPTIONS: --add-opens=java.base/javax.security.auth=ALL-UNNAMED
openjdk version "17.0.19" 2026-04-21
OpenJDK Runtime Environment Temurin-17.0.19+10 (build 17.0.19+10)
OpenJDK 64-Bit Server VM Temurin-17.0.19+10 (build 17.0.19+10, mixed mode, sharing)

winutils.exe found: True
hadoop.dll found:   True


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, StringType, TimestampType
 
spark = (
    SparkSession.builder
    .appName("nyc-yellow-taxi-2025-phase1")
    .master("local[*]")                          # use all CPU cores on your machine
    .config("spark.driver.memory", "6g")          # increase to "6g" if you have 16GB RAM
    .config("spark.sql.shuffle.partitions", "8")  # default 200 is overkill locally
    .config("spark.sql.session.timeZone", "America/New_York")
    .config("spark.sql.parquet.datetimeRebaseModeInRead", "CORRECTED")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")   # stops Spark shouting INFO logs in your console
print("Spark version:", spark.version)
print("✓ Session ready")

Spark version: 3.5.1
✓ Session ready


In [3]:
DATA_DIR = "NYC_Data/rawData"
 
df_raw = spark.read.parquet(f"{DATA_DIR}/yellow_tripdata_2025-01.parquet")
 
print("=" * 55)
print("RAW DATA — January 2025")
print("=" * 55)
print(f"Rows    : {df_raw.count():,}")
print(f"Columns : {len(df_raw.columns)}")

RAW DATA — January 2025
Rows    : 3,475,226
Columns : 20


In [4]:
 
print("=" * 55)
print("STEP 1 · Schema (column names & data types)")
print("=" * 55)
df_raw.printSchema()
 
print("=" * 55)
print("STEP 2 · First 5 rows (so you can eyeball the data)")
print("=" * 55)
df_raw.show(5, truncate=False)
 
print("=" * 55)
print("STEP 3 · Missing value count per column")
print("=" * 55)
total = df_raw.count()
null_df = df_raw.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_raw.columns]
)
null_row = null_df.collect()[0].asDict()
print(f"{'Column':<30} {'Nulls':>10}  {'%':>6}")
print("-" * 50)
for col, n in null_row.items():
    pct = n / total * 100
    flag = " ⚠️" if pct > 5 else ""
    print(f"{col:<30} {n:>10,}  {pct:>5.1f}%{flag}")
 
print()
print("=" * 55)
print("STEP 4 · Key column value ranges")
print("=" * 55)
# These tell you what 'illegal' values exist before cleaning
df_raw.select("trip_distance", "fare_amount", "total_amount",
              "passenger_count").summary(
    "count", "min", "max", "mean", "50%"
).show()
 
print("Unique VendorID values     :", [r[0] for r in df_raw.select("VendorID").distinct().collect()])
print("Unique RatecodeID values   :", sorted([r[0] for r in df_raw.select("RatecodeID").distinct().collect() if r[0] is not None]))
print("Unique payment_type values :", sorted([r[0] for r in df_raw.select("payment_type").distinct().collect()]))
print("Unique store_and_fwd_flag  :", [r[0] for r in df_raw.select("store_and_fwd_flag").distinct().collect() if r[0] is not None])
 
print()
print("Date range in this file:")
df_raw.select(
    F.min("tpep_pickup_datetime").alias("earliest_pickup"),
    F.max("tpep_pickup_datetime").alias("latest_pickup")
).show(truncate=False)
 
# Specific problem counts — document every issue you'll fix
print("=" * 55)
print("STEP 5 · Known data quality issues (raw counts)")
print("=" * 55)
issues = df_raw.agg(
    F.count(F.when(F.col("fare_amount") < 0,          True)).alias("negative_fare"),
    F.count(F.when(F.col("trip_distance") <= 0,       True)).alias("zero_or_neg_distance"),
    F.count(F.when(F.col("passenger_count") == 0,     True)).alias("zero_passengers"),
    F.count(F.when(F.col("passenger_count").isNull(),  True)).alias("null_passengers"),
    F.count(F.when(F.col("RatecodeID") == 99,         True)).alias("ratecode_99_unknown"),
    F.count(F.when(F.col("payment_type") == 0,        True)).alias("payment_type_0_unknown"),
    F.count(F.when(F.col("Airport_fee") < 0,          True)).alias("negative_airport_fee"),
    F.count(F.when(F.col("cbd_congestion_fee") < 0,   True)).alias("negative_cbd_fee"),
    F.count(F.when(
        F.col("tpep_pickup_datetime") >= F.col("tpep_dropoff_datetime"), True
    )).alias("pickup_after_dropoff"),
    F.count(F.when(
        F.col("tpep_pickup_datetime") < "2025-01-01", True
    )).alias("pickup_before_jan"),
    F.count(F.when(
        F.col("tpep_pickup_datetime") > "2025-01-31 23:59:59", True
    )).alias("pickup_after_jan"),
)
issues.show(vertical=True)

STEP 1 · Schema (column names & data types)
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

STEP 2 · First 5 rows (so you c

In [5]:
print("=" * 55)
print("CELL 4 · Fixing data types")
print("=" * 55)
 
df = (
    df_raw
    # --- Integer IDs ---
    .withColumn("VendorID",        F.col("VendorID").cast(IntegerType()))
    .withColumn("RatecodeID",      F.col("RatecodeID").cast(IntegerType()))
    .withColumn("PULocationID",    F.col("PULocationID").cast(IntegerType()))
    .withColumn("DOLocationID",    F.col("DOLocationID").cast(IntegerType()))
    .withColumn("payment_type",    F.col("payment_type").cast(IntegerType()))
    .withColumn("passenger_count", F.col("passenger_count").cast(IntegerType()))
 
    # --- Timestamps (already correct in parquet, recast to be safe) ---
    .withColumn("tpep_pickup_datetime",  F.col("tpep_pickup_datetime").cast(TimestampType()))
    .withColumn("tpep_dropoff_datetime", F.col("tpep_dropoff_datetime").cast(TimestampType()))
 
    # --- Money / distance columns ---
    .withColumn("trip_distance",         F.col("trip_distance").cast(DoubleType()))
    .withColumn("fare_amount",           F.col("fare_amount").cast(DoubleType()))
    .withColumn("extra",                 F.col("extra").cast(DoubleType()))
    .withColumn("mta_tax",               F.col("mta_tax").cast(DoubleType()))
    .withColumn("tip_amount",            F.col("tip_amount").cast(DoubleType()))
    .withColumn("tolls_amount",          F.col("tolls_amount").cast(DoubleType()))
    .withColumn("improvement_surcharge", F.col("improvement_surcharge").cast(DoubleType()))
    .withColumn("total_amount",          F.col("total_amount").cast(DoubleType()))
    .withColumn("congestion_surcharge",  F.col("congestion_surcharge").cast(DoubleType()))
    .withColumn("Airport_fee",           F.col("Airport_fee").cast(DoubleType()))
    .withColumn("cbd_congestion_fee",    F.col("cbd_congestion_fee").cast(DoubleType()))
 
    # --- String ---
    .withColumn("store_and_fwd_flag", F.col("store_and_fwd_flag").cast(StringType()))
)
 
print("✓ All columns cast to correct types")
df.printSchema()

CELL 4 · Fixing data types
✓ All columns cast to correct types
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [6]:
print("=" * 55)
print("CELL 5 · Handling missing values")
print("=" * 55)
 
# --- 5a. DROP rows where timestamps or location IDs are null ---
# Reason: a trip with no time or location is completely unusable.
before = df.count()
df = df.dropna(subset=[
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID"
])
print(f"  Dropped rows (null timestamps / location): {before - df.count():,}")
 
# --- 5b. FILL surcharge columns with 0 ---
# Reason: VendorID 6 & 7 (newer vendors) don't report these — absence means $0
df = df.fillna({
    "congestion_surcharge" : 0.0,
    "Airport_fee"          : 0.0,
    "extra"                : 0.0,
    "mta_tax"              : 0.0,
    "tip_amount"           : 0.0,
    "tolls_amount"         : 0.0,
    "improvement_surcharge": 0.0,
})
print("  ✓ Filled surcharge nulls → 0.0  (vendor 6/7 don't report these)")
 
# --- 5c. FILL passenger_count with 1 ---
# Reason: 1 is the overwhelming mode; the null pattern matches vendor 6/7 trips
df = df.fillna({"passenger_count": 1})
print("  ✓ Filled passenger_count nulls → 1  (most common value)")
 
# --- 5d. FILL categorical IDs with -1 (Unknown sentinel) ---
# We keep these rows — they still have valid fare & location data
df = df.fillna({"VendorID": -1, "RatecodeID": -1})
print("  ✓ Filled VendorID / RatecodeID nulls → -1  (Unknown sentinel)")
 
# --- 5e. Recode payment_type 0 (Unknown) → -1 to standardise ---
# payment_type 0 always co-occurs with vendor 6/7 null block
df = df.withColumn(
    "payment_type",
    F.when(F.col("payment_type") == 0, -1).otherwise(F.col("payment_type"))
)
print("  ✓ Recoded payment_type 0 → -1  (Unknown, matches vendor 6/7 pattern)")
 
# --- 5f. FILL store_and_fwd_flag with 'N' ---
df = df.fillna({"store_and_fwd_flag": "N"})
print("  ✓ Filled store_and_fwd_flag nulls → 'N'")
 
# Verify — should be zero remaining nulls (except any newly cast failures)
print()
print("  Remaining nulls after fill:")
null_check = df.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]
).collect()[0].asDict()
remaining = {k: v for k, v in null_check.items() if v > 0}
if remaining:
    for col, n in remaining.items():
        print(f"    {col}: {n:,}")
else:
    print("    ✓ None — all nulls resolved")
 

CELL 5 · Handling missing values
  Dropped rows (null timestamps / location): 0
  ✓ Filled surcharge nulls → 0.0  (vendor 6/7 don't report these)
  ✓ Filled passenger_count nulls → 1  (most common value)
  ✓ Filled VendorID / RatecodeID nulls → -1  (Unknown sentinel)
  ✓ Recoded payment_type 0 → -1  (Unknown, matches vendor 6/7 pattern)
  ✓ Filled store_and_fwd_flag nulls → 'N'

  Remaining nulls after fill:
    ✓ None — all nulls resolved


In [7]:
print("=" * 55)
print("CELL 6 · Removing duplicates")
print("=" * 55)
 
KEY_COLS = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "total_amount"
]
 
before = df.count()
df = df.dropDuplicates(KEY_COLS)
removed = before - df.count()
print(f"  Rows before : {before:,}")
print(f"  Duplicates  : {removed:,}")
print(f"  Rows after  : {df.count():,}")

CELL 6 · Removing duplicates
  Rows before : 3,475,226
  Duplicates  : 1
  Rows after  : 3,475,225


In [8]:
print("=" * 55)
print("CELL 7 · Validating & filtering bad data")
print("=" * 55)
 
def filter_log(df, condition, label):
    """Apply a filter and print how many rows it removed."""
    before = df.count()
    df = df.filter(condition)
    removed = before - df.count()
    print(f"  {label:<52} removed: {removed:>7,}")
    return df
 
# --- 7a. Compute trip duration first (we need it for filtering) ---
df = df.withColumn(
    "trip_duration_minutes",
    ((F.unix_timestamp("tpep_dropoff_datetime") -
      F.unix_timestamp("tpep_pickup_datetime")) / 60).cast(DoubleType())
)
 
# --- 7b. Apply all filters ---
# Timestamps
df = filter_log(df,
    F.col("tpep_pickup_datetime") < F.col("tpep_dropoff_datetime"),
    "Pickup must be BEFORE drop-off")
 
df = filter_log(df,
    (F.col("tpep_pickup_datetime") >= "2025-01-01") &
    (F.col("tpep_pickup_datetime") <= "2025-01-31 23:59:59"),
    "Pickup date within January 2025")
 
df = filter_log(df,
    (F.col("trip_duration_minutes") >= 1) &
    (F.col("trip_duration_minutes") <= 1440),   # max 24 hours
    "Trip duration: 1 min – 24 hours")
 
# Distance
df = filter_log(df,
    (F.col("trip_distance") > 0) &
    (F.col("trip_distance") <= 500),             # > 500 miles is clearly wrong
    "Trip distance: > 0 and ≤ 500 miles")
 
# Money — negatives are refunds/corrections logged in error
df = filter_log(df,
    (F.col("fare_amount") >= 0) &
    (F.col("fare_amount") <= 50_000),
    "Fare amount: $0 – $50,000")
 
df = filter_log(df,
    F.col("total_amount") >= 0,
    "Total amount: ≥ $0")
 
# We found negative airport fees and CBD fees — filter those
df = filter_log(df,
    F.col("Airport_fee") >= 0,
    "Airport fee: ≥ $0")
 
df = filter_log(df,
    F.col("cbd_congestion_fee") >= 0,
    "CBD congestion fee: ≥ $0")
 
# Passengers
df = filter_log(df,
    (F.col("passenger_count") >= 1) &
    (F.col("passenger_count") <= 9),             # NYC TLC max = 9
    "Passenger count: 1 – 9")
 
# --- 7c. Recode (not filter) invalid category codes ---
# RatecodeID 99 = 'Unknown' in TLC data dictionary → recode to -1
df = df.withColumn(
    "RatecodeID",
    F.when(F.col("RatecodeID") == 99, -1).otherwise(F.col("RatecodeID"))
)
print(f"  {'Recoded RatecodeID 99 → -1 (Unknown)':<52} (no rows dropped)")

CELL 7 · Validating & filtering bad data
  Pickup must be BEFORE drop-off                       removed:   2,051
  Pickup date within January 2025                      removed:      22
  Trip duration: 1 min – 24 hours                      removed:  38,080
  Trip distance: > 0 and ≤ 500 miles                   removed:  64,494
  Fare amount: $0 – $50,000                            removed: 126,932
  Total amount: ≥ $0                                   removed:     147
  Airport fee: ≥ $0                                    removed:       0
  CBD congestion fee: ≥ $0                             removed:       0
  Passenger count: 1 – 9                               removed:  23,537
  Recoded RatecodeID 99 → -1 (Unknown)                 (no rows dropped)


In [9]:
print("=" * 55)
print("CELL 8 · Adding derived columns")
print("=" * 55)
 
df = (df
    # ── Time features ──────────────────────────────────────
    .withColumn("pickup_date",       F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_year",       F.year("tpep_pickup_datetime"))
    .withColumn("pickup_month",      F.month("tpep_pickup_datetime"))
    .withColumn("pickup_day",        F.dayofmonth("tpep_pickup_datetime"))
    .withColumn("pickup_hour",       F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_dayofweek",  F.dayofweek("tpep_pickup_datetime"))  # 1=Sun, 7=Sat
    .withColumn("pickup_dayname",    F.date_format("tpep_pickup_datetime", "EEEE"))
    .withColumn("is_weekend",
        F.when(F.dayofweek("tpep_pickup_datetime").isin([1, 7]), True).otherwise(False)
    )
    .withColumn("time_of_day",
        F.when(F.col("pickup_hour").between(5,  11), "Morning")
         .when(F.col("pickup_hour").between(12, 16), "Afternoon")
         .when(F.col("pickup_hour").between(17, 20), "Evening")
         .when(F.col("pickup_hour").between(21, 23), "Night")
         .otherwise("Late Night")                                           # 0–4 AM
    )
 
    # ── Speed ───────────────────────────────────────────────
    .withColumn("avg_speed_mph",
        F.when(F.col("trip_duration_minutes") > 0,
            F.round(F.col("trip_distance") / (F.col("trip_duration_minutes") / 60), 2)
        ).otherwise(None)
    )
 
    # ── Cost per mile ────────────────────────────────────────
    .withColumn("fare_per_mile",
        F.when(F.col("trip_distance") > 0,
            F.round(F.col("fare_amount") / F.col("trip_distance"), 2)
        ).otherwise(None)
    )
 
    # ── Tip percentage ───────────────────────────────────────
    .withColumn("tip_pct",
        F.when(F.col("fare_amount") > 0,
            F.round(F.col("tip_amount") / F.col("fare_amount") * 100, 2)
        ).otherwise(0.0)
    )
 
    # ── Human-readable labels ────────────────────────────────
    .withColumn("payment_label",
        F.when(F.col("payment_type") == 1,  "Credit Card")
         .when(F.col("payment_type") == 2,  "Cash")
         .when(F.col("payment_type") == 3,  "No Charge")
         .when(F.col("payment_type") == 4,  "Dispute")
         .when(F.col("payment_type") == 5,  "Unknown")
         .otherwise("Unknown")
    )
    .withColumn("ratecode_label",
        F.when(F.col("RatecodeID") == 1,  "Standard")
         .when(F.col("RatecodeID") == 2,  "JFK")
         .when(F.col("RatecodeID") == 3,  "Newark")
         .when(F.col("RatecodeID") == 4,  "Nassau/Westchester")
         .when(F.col("RatecodeID") == 5,  "Negotiated")
         .when(F.col("RatecodeID") == 6,  "Group Ride")
         .otherwise("Unknown")
    )
)
 
print("  Added columns:")
new_cols = ["pickup_date","pickup_year","pickup_month","pickup_day",
            "pickup_hour","pickup_dayofweek","pickup_dayname","is_weekend",
            "time_of_day","trip_duration_minutes","avg_speed_mph",
            "fare_per_mile","tip_pct","payment_label","ratecode_label"]
for c in new_cols:
    print(f"    • {c}")

CELL 8 · Adding derived columns
  Added columns:
    • pickup_date
    • pickup_year
    • pickup_month
    • pickup_day
    • pickup_hour
    • pickup_dayofweek
    • pickup_dayname
    • is_weekend
    • time_of_day
    • trip_duration_minutes
    • avg_speed_mph
    • fare_per_mile
    • tip_pct
    • payment_label
    • ratecode_label


In [10]:
print()
print("=" * 55)
print("CELL 9 · Final Quality Report — January 2025")
print("=" * 55)
 
raw_count     = 3_475_226          # from Cell 2 above
cleaned_count = df.count()
 
print(f"  Raw rows         : {raw_count:>10,}")
print(f"  Cleaned rows     : {cleaned_count:>10,}")
print(f"  Rows removed     : {raw_count - cleaned_count:>10,}  ({(raw_count - cleaned_count)/raw_count*100:.1f}%)")
print(f"  Rows retained    : {cleaned_count/raw_count*100:.1f}%")
 
print()
print("  Key stats (cleaned):")
df.select("trip_distance","trip_duration_minutes","fare_amount",
          "total_amount","avg_speed_mph","tip_pct").summary(
    "count","mean","min","50%","max"
).show()
 
print("  Payment type breakdown:")
df.groupBy("payment_label").count().orderBy("count", ascending=False).show()
 
print("  Time of day breakdown:")
df.groupBy("time_of_day").count().orderBy("count", ascending=False).show()
 
print("  Ratecode breakdown:")
df.groupBy("ratecode_label").count().orderBy("count", ascending=False).show()
 
print("  Weekend vs Weekday:")
df.groupBy("is_weekend").count().show()
 


CELL 9 · Final Quality Report — January 2025
  Raw rows         :  3,475,226
  Cleaned rows     :  3,219,962
  Rows removed     :    255,264  (7.3%)
  Rows retained    : 92.7%

  Key stats (cleaned):
+-------+------------------+---------------------+------------------+------------------+------------------+------------------+
|summary|     trip_distance|trip_duration_minutes|       fare_amount|      total_amount|     avg_speed_mph|           tip_pct|
+-------+------------------+---------------------+------------------+------------------+------------------+------------------+
|  count|           3219962|              3219962|           3219962|           3219962|           3219962|           3219962|
|   mean|3.1830952725529325|   15.183388685953558|17.936870782946148|26.803445121403055|11.338697611959256|19.815202424744076|
|    min|              0.01|                  1.0|               0.0|               0.0|               0.0|               0.0|
|    50%|               1.7|   11.733

In [11]:
print("=" * 55)
print("CELL 10 · Exporting cleaned January data")
print("=" * 55)
 
OUTPUT_DIR = "nyc_taxi_cleaned"
 
# Save as partitioned parquet (fast for EDA later)
df.write.mode("overwrite").parquet(f"{OUTPUT_DIR}/jan_2025_cleaned")
print(f"  ✓ Parquet saved → {OUTPUT_DIR}/jan_2025_cleaned/")
 
# Save a 5,000-row CSV sample for quick preview in Excel/Sheets
(df.limit(5000)
   .toPandas()
   .to_csv(f"{OUTPUT_DIR}/jan_2025_sample_5000.csv", index=False))
print(f"  ✓ CSV sample (5k rows) → {OUTPUT_DIR}/jan_2025_sample_5000.csv")
 
print()
print("✅ Phase 1 complete! January is clean.")
print("   Now open Phase 2 notebook to run all 12 months.")
 
spark.stop()

CELL 10 · Exporting cleaned January data
  ✓ Parquet saved → nyc_taxi_cleaned/jan_2025_cleaned/
  ✓ CSV sample (5k rows) → nyc_taxi_cleaned/jan_2025_sample_5000.csv

✅ Phase 1 complete! January is clean.
   Now open Phase 2 notebook to run all 12 months.
